# 03 — Model Evaluation: GloVe vs MiniLM

This notebook benchmarks the two retrieval models against each other using standard IR metrics:
- **Precision@5** and **Precision@10** — what fraction of the top-k results are actually relevant?
- **MRR (Mean Reciprocal Rank)** — on average, how high does the first relevant tweet appear?

**Prerequisites:** Run `01_build_index.ipynb` first.

**Note on the test set:** The relevance judgments below were created manually. Each query has a small set of tweet indices that were judged relevant by inspection. In a real IR system you'd crowdsource this — for a portfolio project, 25 queries with binary relevance is standard practice (e.g. MS MARCO uses the same approach).

In [ ]:
# --- STEP 0: INSTALL DEPENDENCIES ---
!pip install faiss-cpu sentence-transformers --quiet

In [ ]:
# --- STEP 1: IMPORTS ---
import json
import os
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("Imports ready.")

In [ ]:
# --- STEP 2: LOAD INDEXES AND TWEETS ---

base = '/content/' if os.path.exists('/content/minilm_faiss.index') else ''

print("Loading FAISS indexes...")
minilm_index = faiss.read_index(f'{base}minilm_faiss.index')
glove_index  = faiss.read_index(f'{base}glove_faiss.index')

with open(f'{base}tweets.json', 'r', encoding='utf-8') as f:
    tweets = json.load(f)

print(f"MiniLM index: {minilm_index.ntotal:,} vectors")
print(f"GloVe index:  {glove_index.ntotal:,} vectors")
print(f"Tweets loaded: {len(tweets):,}")

In [ ]:
# --- STEP 3: LOAD ENCODERS ---

print("Loading MiniLM encoder...")
minilm_encoder = SentenceTransformer('all-MiniLM-L6-v2')

print("Loading GloVe encoder...")
glove_encoder = SentenceTransformer('average_word_embeddings_glove.840B.300d')

print("Both encoders loaded.")

In [ ]:
# --- STEP 4: SEARCH FUNCTIONS ---

def search_minilm(query, top_k=10):
    """
    Encode a query with MiniLM and retrieve top-k tweet indices from the FAISS IP index.

    Parameters:
        query (str): The search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by cosine similarity (highest first).
    """
    vec = minilm_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    # Normalize to match how the index was built
    vec = vec / np.linalg.norm(vec)
    _, indices = minilm_index.search(vec, top_k)
    return indices[0].tolist()


def search_glove(query, top_k=10):
    """
    Encode a query with GloVe and retrieve top-k tweet indices from the FAISS L2 index.

    Parameters:
        query (str): The search query.
        top_k (int): Number of results to return.

    Returns:
        list[int]: Tweet indices ordered by L2 distance (closest first).
    """
    vec = glove_encoder.encode([query], convert_to_numpy=True).astype(np.float32)
    _, indices = glove_index.search(vec, top_k)
    return indices[0].tolist()


print("Search functions defined.")

In [ ]:
# --- STEP 5: TEST SET ---
#
# Each tuple: (query_string, set_of_relevant_tweet_indices)
#
# Relevance judgments were determined by keyword search + manual inspection of the
# raw tweet data. Binary relevance: 1 if the tweet is clearly about the query topic.
# Indices refer to position in tweets.json (built by 01_build_index.ipynb).

TEST_SET = [
    ("looking for a job",
        {276, 496, 1298, 1335, 1604, 1612, 1633, 1684}),
    # All are #hiring / job-posting tweets mentioning specific openings

    ("I got fired today",
        {39145, 47191, 49915, 59279, 89940}),
    # Tweets about people being fired or laid off from jobs

    ("climate change is real",
        {1154, 3332, 4450, 5618, 8897, 9407, 10513}),
    # Tweets explicitly about climate change / global warming as a topic

    ("global warming effects",
        {1295, 14823}),
    # Tweets about specific effects of warming (drought, rising temps)
    # Intentionally small — most climate tweets in this dataset are opinion, not effects

    ("feeling happy and excited",
        {30, 1095, 1113, 2018, 2209, 2465}),
    # Tweets expressing personal happiness or excitement

    ("sad and depressed",
        {4459, 5143, 7513, 7640}),
    # Tweets about feeling sad, depressed, or heartbroken

    ("watching a movie tonight",
        {6485, 7418, 7510, 16644, 23145}),
    # Tweets about movie nights or watching films

    ("sports game results",
        {12, 134, 610, 634, 989, 2029}),
    # Tweets about playoff games, scores, and match outcomes

    ("eating delicious food",
        {2164, 4010, 7941}),
    # Tweets about food tasting good or enjoying a meal

    ("health and fitness tips",
        {2584, 4718, 9090, 9122, 14588, 15234}),
    # Tweets about gym visits, healthy eating, working out

    ("political election news",
        {132, 347, 515, 551, 749}),
    # Tweets about elections, voting, political figures

    ("new music release",
        {1897, 3243, 4662}),
    # Tweets about new songs, albums, or music playlists

    ("morning coffee routine",
        {1913, 1968, 16130, 18528, 33465, 34452}),
    # Tweets about needing or enjoying morning coffee

    ("travel plans and vacation",
        {3202, 3627, 5398, 6764, 7404}),
    # Tweets about upcoming trips, vacations, or travel plans

    ("technology and smartphones",
        {15943, 18729, 19779, 20927, 21625, 21770}),
    # Tweets about phones, iPhone/Android, and tech products

    ("social media addiction",
        {520, 2192}),
    # Tweets about social media habits — small set, topic underrepresented in dataset

    ("family and relationships",
        {106, 363, 469, 534, 794}),
    # Tweets mentioning parents, siblings, or family dynamics

    ("studying for exams",
        {936, 7605, 8317, 9634, 12779, 13927}),
    # Tweets about studying, exam stress, or cramming

    ("funny jokes and memes",
        {1005, 1354}),
    # Tweets that are clearly comedic (beyond casual lmao use)

    ("news headlines today",
        {1356, 1386, 2907, 4404}),
    # Tweets with "breaking news" or referencing current news stories

    ("feeling tired and exhausted",
        {663, 855, 1032, 1602, 1993, 2123}),
    # Tweets about tiredness, needing sleep, or feeling drained

    ("workout at the gym",
        {398, 9090, 15234, 21486, 24379, 29390, 35356}),
    # Tweets about gym sessions, leg day, cardio, lifting

    ("birthday celebration",
        {94, 578, 702, 715, 759, 951, 1015, 1700}),
    # Tweets wishing someone happy birthday or celebrating one

    ("money and financial stress",
        {1917, 7402, 8754}),
    # Tweets about being unable to afford things or financial pressure

    ("dogs and pets",
        {288, 563, 739, 1214, 4006, 6852, 7673}),
    # Tweets about dogs, cats, or pet behavior
]

print(f"Test set: {len(TEST_SET)} queries, all relevance judgments filled in.")
print(f"Total judged-relevant tweets: {sum(len(r) for _, r in TEST_SET)}")
print("Ready to run evaluation — proceed to the metric cells.")

In [ ]:
# --- HELPER: INSPECT RESULTS FOR A QUERY ---
# Use this cell to browse top results and decide which indices to mark as relevant.
# Change the query and run to inspect different topics.

INSPECT_QUERY = "looking for a job"
TOP_N = 20

minilm_results = search_minilm(INSPECT_QUERY, top_k=TOP_N)
glove_results  = search_glove(INSPECT_QUERY, top_k=TOP_N)

print(f"Query: '{INSPECT_QUERY}'")
print(f"\n{'MiniLM Results':^60} | {'GloVe Results':^60}")
print("-" * 130)

for i in range(TOP_N):
    m_idx = minilm_results[i]
    g_idx = glove_results[i]
    m_text = tweets[m_idx][:55].replace('\n', ' ')
    g_text = tweets[g_idx][:55].replace('\n', ' ')
    print(f"  [{m_idx:6d}] {m_text:<55} | [{g_idx:6d}] {g_text}")

In [ ]:
# --- STEP 6: METRIC FUNCTIONS ---

def precision_at_k(retrieved, relevant, k):
    """
    Compute Precision@k: the fraction of the top-k results that are relevant.

    Parameters:
        retrieved (list[int]): Ranked list of retrieved tweet indices.
        relevant (set[int]): Set of tweet indices judged relevant for this query.
        k (int): Cutoff rank.

    Returns:
        float: Precision@k score in [0, 1]. Returns 0 if relevant set is empty.
    """
    if not relevant:
        return 0.0
    top_k = retrieved[:k]
    hits = sum(1 for idx in top_k if idx in relevant)
    return hits / k


def reciprocal_rank(retrieved, relevant):
    """
    Compute the Reciprocal Rank for a single query: 1/rank of the first relevant result.

    If no relevant tweet appears in the retrieved list, returns 0.
    MRR averages this across all queries — a higher MRR means relevant results
    appear closer to the top on average.

    Parameters:
        retrieved (list[int]): Ranked list of retrieved tweet indices.
        relevant (set[int]): Set of tweet indices judged relevant.

    Returns:
        float: Reciprocal rank score in (0, 1], or 0 if no relevant result found.
    """
    for rank, idx in enumerate(retrieved, start=1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0


print("Metric functions defined.")

In [ ]:
# --- STEP 7: RUN EVALUATION ---
# Skip queries with empty relevance sets (not yet judged)

judged_queries = [(q, rel) for q, rel in TEST_SET if len(rel) > 0]

if len(judged_queries) == 0:
    print("No relevance judgments found in TEST_SET yet.")
    print("Use the inspection helper cell above to browse tweets and fill in the relevant indices.")
else:
    minilm_p5, minilm_p10, minilm_rr = [], [], []
    glove_p5,  glove_p10,  glove_rr  = [], [], []

    print(f"Evaluating {len(judged_queries)} judged queries...")

    for query, relevant in judged_queries:
        m_results = search_minilm(query, top_k=10)
        g_results = search_glove(query, top_k=10)

        minilm_p5.append(precision_at_k(m_results, relevant, 5))
        minilm_p10.append(precision_at_k(m_results, relevant, 10))
        minilm_rr.append(reciprocal_rank(m_results, relevant))

        glove_p5.append(precision_at_k(g_results, relevant, 5))
        glove_p10.append(precision_at_k(g_results, relevant, 10))
        glove_rr.append(reciprocal_rank(g_results, relevant))

    print("Evaluation complete.")

In [ ]:
# --- STEP 8: PRINT COMPARISON TABLE ---

if len(judged_queries) == 0:
    print("No results to display — fill in TEST_SET first.")
else:
    def avg(lst):
        return sum(lst) / len(lst) if lst else 0.0

    m_p5  = avg(minilm_p5)
    m_p10 = avg(minilm_p10)
    m_mrr = avg(minilm_rr)
    g_p5  = avg(glove_p5)
    g_p10 = avg(glove_p10)
    g_mrr = avg(glove_rr)

    # Compute relative improvement of MiniLM over GloVe
    def pct_diff(a, b):
        """Percentage difference of a over b. Returns 'N/A' if b is zero."""
        return f"+{(a - b) / b * 100:.1f}%" if b > 0 else "N/A"

    print(f"\n{'Metric':<15} {'MiniLM':>10} {'GloVe':>10} {'MiniLM vs GloVe':>18}")
    print("-" * 55)
    print(f"{'Precision@5':<15} {m_p5:>10.4f} {g_p5:>10.4f} {pct_diff(m_p5, g_p5):>18}")
    print(f"{'Precision@10':<15} {m_p10:>10.4f} {g_p10:>10.4f} {pct_diff(m_p10, g_p10):>18}")
    print(f"{'MRR':<15} {m_mrr:>10.4f} {g_mrr:>10.4f} {pct_diff(m_mrr, g_mrr):>18}")
    print(f"\nEvaluated on {len(judged_queries)} queries.")

    if m_mrr > g_mrr:
        print(f"\nResult: MiniLM outperformed GloVe by {(m_mrr - g_mrr) / g_mrr * 100:.1f}% on MRR.")
    elif g_mrr > m_mrr:
        print(f"\nResult: GloVe outperformed MiniLM by {(g_mrr - m_mrr) / m_mrr * 100:.1f}% on MRR.")
    else:
        print("\nResult: Both models achieved identical MRR.")

In [ ]:
# --- BONUS: PER-QUERY BREAKDOWN ---
# Useful for seeing which query types favor MiniLM vs GloVe

if len(judged_queries) > 0:
    print(f"\n{'Query':<40} {'MiniLM MRR':>12} {'GloVe MRR':>12} {'Winner':>10}")
    print("-" * 78)
    for i, (query, _) in enumerate(judged_queries):
        m = minilm_rr[i]
        g = glove_rr[i]
        winner = 'MiniLM' if m > g else ('GloVe' if g > m else 'Tie')
        print(f"  {query[:38]:<40} {m:>12.4f} {g:>12.4f} {winner:>10}")